# SECFSN 05 · transparent decision systems

**Layer 3B value proposition:** convert upstream SEC-derived evidence into deterministic, auditable research-decision states without retraining a model or changing the source evidence.

### What this 2-minute demo shows

- The prepared evidence panel that this repository consumes.
- The deterministic rules that synthesize risk and detect overrides.
- The resulting long-side, short-side, or monitor research state.

`secfsn_plus_decisionsystems` is **SECFSN 05 / Layer 3B**. SECFSN 01–04 form its upstream analytical stack: normalized SEC facts, derived financial features, structural evidence, and optional model-derived evidence. SECFSN 05 owns the governed decision rules, synthesis, veto/override handling, screening APIs, and auditable research-state output; it does not train predictive models.

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


def find_repo_root(start: Path) -> Path:
    """Locate a secfsn_plus_decisionsystems checkout from this notebook."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "secfsn_plus_decisionsystems").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from a secfsn_plus_decisionsystems checkout.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from secfsn_plus_decisionsystems.integration.decision_engine import run_decision_engine
from secfsn_plus_decisionsystems.integration.model_bridge import merge_model_outputs
from secfsn_plus_decisionsystems.screening.risk_filtered_longs import screen_risk_filtered_longs
from secfsn_plus_decisionsystems.screening.short_candidates import screen_short_candidates


TABLE_STYLES = [
    {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "12px")]},
    {"selector": "th", "props": [("background-color", "#14324A"), ("color", "white"), ("padding", "7px 10px"), ("text-align", "left")]},
    {"selector": "td", "props": [("padding", "6px 10px"), ("border-bottom", "1px solid #D9E2EA")]},
]


def show_table(frame: pd.DataFrame, *, score_columns: tuple[str, ...] = ()) -> None:
    formatter = {column: "{:.3f}" for column in score_columns if column in frame.columns}
    display(frame.style.format(formatter, na_rep="—").set_table_styles(TABLE_STYLES))


architecture_view = pd.DataFrame(
    [
        ("01", "Layer 1", "secfsn", "Normalized SEC facts"),
        ("02", "Layer 2", "secfsn_plus", "Derived financial features"),
        ("03", "Layer 2.5", "secfsn_plus_structural", "Structural evidence"),
        ("04", "Layer 3A", "secfsn_plus_models", "Model-derived evidence"),
        ("05", "Layer 3B", "secfsn_plus_decisionsystems", "Governed decision outputs"),
    ],
    columns=["SECFSN", "Layer", "Repository", "Role"],
)


def highlight_current_repository(row: pd.Series) -> list[str]:
    return ["background-color: #E5F4EA; color: #14532D; font-weight: 650"] * len(row) if row["SECFSN"] == "05" else [""] * len(row)


display(architecture_view.style.apply(highlight_current_repository, axis=1).set_table_styles(TABLE_STYLES))
print("SECFSN 05 / Layer 3B demo ready — deterministic fixture, no network calls, credentials, or model training.")

,SECFSN,Layer,Repository,Role
0,01,Layer 1,secfsn,Normalized SEC facts
1,02,Layer 2,secfsn_plus,Derived financial features
2,03,Layer 2.5,secfsn_plus_structural,Structural evidence
3,04,Layer 3A,secfsn_plus_models,Model-derived evidence
4,05,Layer 3B,secfsn_plus_decisionsystems,Governed decision outputs


SECFSN 05 / Layer 3B demo ready — deterministic fixture, no network calls, credentials, or model training.


## 1. Upstream evidence arrives as a prepared panel

The committed fixture is a small, synthetic `(cik, period)` panel built for tests. It contains precomputed structural/accounting signals: a scaled equity/net-income residual, a scaled cash-versus-earnings gap, and accrual intensity. SECFSN 01 provides normalized SEC facts, SECFSN 02 adds derived financial features, and SECFSN 03 adds structural accounting and filing-pattern evidence. SECFSN 05 consumes the available evidence; those upstream repositories own its derivation.

SECFSN 04 / Layer 3A (`secfsn_plus_models`) is the supported upstream source for validated model-output contracts and model-derived evidence such as `model_risk_probability`. This committed SECFSN 05 fixture does **not** include an actual SECFSN 04 trained-model artifact. The bridge therefore receives `None`, preserves the panel, and displays missing model evidence rather than silently fabricating or weighting a probability.

In [2]:
fixture_path = REPO_ROOT / "tests" / "fixtures" / "upstream_prepared_panel.csv"
prepared_panel = pd.read_csv(fixture_path, dtype={"cik": str, "period": str}).set_index(["cik", "period"]).sort_index()
decision_input = merge_model_outputs(prepared_panel, None)
assert decision_input.equals(prepared_panel)

evidence_view = prepared_panel.loc[:, [
    "equity_ni_residual_scaled",
    "cash_vs_earnings_gap_scaled",
    "accrual_intensity",
]].rename(columns={
    "equity_ni_residual_scaled": "Equity / NI residual",
    "cash_vs_earnings_gap_scaled": "Cash / earnings gap",
    "accrual_intensity": "Accrual intensity",
})
evidence_view["SECFSN 04 model evidence"] = "Missing: no fixture artifact"
show_table(evidence_view, score_columns=("Equity / NI residual", "Cash / earnings gap", "Accrual intensity"))

## 2. Deterministic rules synthesize the evidence

SECFSN 05 owns the canonical `run_decision_engine()` workflow. It builds a normalized structural-risk score and fraud-proxy score from available SECFSN 01–03 evidence, combines them into the default hybrid-risk score, and evaluates cross-sectional anomalies, rolling instability, and persistent high risk.

The `risk_override` flag is the SECFSN 05 veto/exception state: it activates when anomaly or persistence rules require an override. With the default configuration, absent SECFSN 04 model evidence does not block the deterministic workflow; a supplied SECFSN 04 probability is an optional diagnostic input and is not weighted into the default hybrid score.

In [3]:
decision_result = run_decision_engine(decision_input)

required_outputs = {
    "structural_risk_score", "fraud_proxy_score", "hybrid_risk_score",
    "long_allowed", "short_allowed", "risk_override",
}
assert required_outputs.issubset(decision_result.columns)
assert decision_result.index.equals(prepared_panel.index)
print("Decision engine complete — the input index is preserved and all canonical decision outputs are present.")

Decision engine complete — the input index is preserved and all canonical decision outputs are present.


## 3. Resulting decision state

The final table keeps the audit trail on one screen: SECFSN 01–03 evidence was present, SECFSN 04 model evidence was missing, SECFSN 05 structural and fraud-proxy rules produced the hybrid score, and the override column reveals whether an SECFSN 05 veto applied. The decision state is obtained through this repository's public long- and short-screening APIs.

For this fixture, both `1001` rows qualify for long-side research, `2002 · 2024Q2` qualifies for short-side research, and `2002 · 2024Q1` remains under review. These states direct further research; they do not assert predictive performance, fraud, or investment suitability.

In [4]:
long_candidates = screen_risk_filtered_longs(decision_result)
short_candidates = screen_short_candidates(decision_result)

decision_state = pd.Series("Monitor", index=decision_result.index, dtype="object")
decision_state.loc[long_candidates.index] = "Long-side research"
decision_state.loc[short_candidates.index] = "Short-side research"

decision_view = (
    decision_result.assign(_decision_state=decision_state)
    .reset_index()
    .assign(
        Record=lambda frame: frame["cik"].astype(str) + " · " + frame["period"],
        **{
            "SECFSN 04 model evidence": "Missing: no fixture artifact",
            "Veto / override": lambda frame: frame["risk_override"].map({True: "Active", False: "None"}),
        },
    )
    [[
        "Record", "structural_risk_score", "fraud_proxy_score", "hybrid_risk_score",
        "SECFSN 04 model evidence", "Veto / override", "_decision_state",
    ]]
    .rename(columns={
        "structural_risk_score": "Structural risk",
        "fraud_proxy_score": "Fraud proxy",
        "hybrid_risk_score": "Hybrid risk",
        "_decision_state": "Decision state",
    })
)

def state_style(value: str) -> str:
    if value == "Short-side research":
        return "background-color: #FDE7E7; color: #8A1C1C; font-weight: 650"
    if value == "Long-side research":
        return "background-color: #E5F4EA; color: #14532D; font-weight: 650"
    return "color: #4B5563"


def veto_style(value: str) -> str:
    return "background-color: #FEF3C7; color: #92400E; font-weight: 650" if value == "Active" else "color: #4B5563"


display(
    decision_view.style
    .format({column: "{:.3f}" for column in ("Structural risk", "Fraud proxy", "Hybrid risk")})
    .map(state_style, subset=["Decision state"])
    .map(veto_style, subset=["Veto / override"])
    .set_table_styles(TABLE_STYLES)
)


,Record,Structural risk,Fraud proxy,Hybrid risk,SECFSN 04 model evidence,Veto / override,Decision state
0,1001 · 2024Q1,0.065,0.070,0.068,Missing: no fixture artifact,None,Long-side research
1,1001 · 2024Q2,0.087,0.090,0.088,Missing: no fixture artifact,None,Long-side research
2,2002 · 2024Q1,0.498,0.495,0.497,Missing: no fixture artifact,None,Monitor
3,2002 · 2024Q2,0.838,0.825,0.832,Missing: no fixture artifact,None,Short-side research
